# Core APIs

This notebook focuses on the most central tidypy capabilities and compares them directly with pure pandas.


## What this example covers

1. Inspect table structure
2. Select columns
3. Apply batch transformations
4. Rename columns in bulk
5. Build conditional labels
6. Summarize by group


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from pandas.testing import assert_frame_equal


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    return start


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from tidypy.tidy import (
    case_when,
    glimpse,
    mutate_across,
    rename_with,
    select,
    starts_with,
    summarize,
)


## 1. Build an employee dataset


In [2]:
df = pd.DataFrame(
    {
        "employee_id": [101, 102, 103, 104, 105],
        "dept": ["Sales", "Sales", "Tech", "Tech", "HR"],
        "score_math": [92.0, None, 88.0, 79.0, None],
        "score_eng": [85.0, 90.0, None, 82.0, 87.0],
        "name": [" Alice ", "Bob ", " Carol", "David", " Frank "],
    }
)

df


,employee_id,dept,score_math,score_eng,name
0,101,Sales,92.0,85.0,Alice
1,102,Sales,NaN,90.0,Bob
2,103,Tech,88.0,NaN,Carol
3,104,Tech,79.0,82.0,David
4,105,HR,NaN,87.0,Frank


## 2. Inspect the structure

This section puts `glimpse()` next to a more typical pandas inspection workflow.


In [3]:
glimpse(df)


Rows: 5, Columns: 5


,column,dtype,non_null,nulls,n_unique,preview
0,employee_id,int64,5,0,5,"101, 102, 103"
1,dept,str,5,0,3,"'Sales', 'Sales', 'Tech'"
2,score_math,float64,3,2,3,"92.0, NA, 88.0"
3,score_eng,float64,4,1,4,"85.0, 90.0, NA"
4,name,str,5,0,5,"' Alice ', 'Bob ', ' Carol'"


,column,dtype,non_null,nulls,n_unique,preview
0,employee_id,int64,5,0,5,"101, 102, 103"
1,dept,str,5,0,3,"'Sales', 'Sales', 'Tech'"
2,score_math,float64,3,2,3,"92.0, NA, 88.0"
3,score_eng,float64,4,1,4,"85.0, 90.0, NA"
4,name,str,5,0,5,"' Alice ', 'Bob ', ' Carol'"


In [4]:
# Roughly equivalent pandas checks
print(df.dtypes)
print() 
print("null counts:")
print(df.isna().sum())
print() 
print("preview:")
df.head(3)


employee_id      int64
dept               str
score_math     float64
score_eng      float64
name               str
dtype: object

null counts:
employee_id    0
dept           0
score_math     2
score_eng      1
name           0
dtype: int64

preview:


,employee_id,dept,score_math,score_eng,name
0,101,Sales,92.0,85.0,Alice
1,102,Sales,NaN,90.0,Bob
2,103,Tech,88.0,NaN,Carol


## 3. Compare column selection

This is mainly about the readability gain from selectors.


In [5]:
pandas_selected = df.loc[:, ["employee_id", "dept", *[c for c in df.columns if c.startswith("score_")]]]
pandas_selected


,employee_id,dept,score_math,score_eng
0,101,Sales,92.0,85.0
1,102,Sales,NaN,90.0
2,103,Tech,88.0,NaN
3,104,Tech,79.0,82.0
4,105,HR,NaN,87.0


In [6]:
tidypy_selected = select(
    df,
    "employee_id",
    "dept",
    starts_with("score_"),
)
tidypy_selected


,employee_id,dept,score_math,score_eng
0,101,Sales,92.0,85.0
1,102,Sales,NaN,90.0
2,103,Tech,88.0,NaN
3,104,Tech,79.0,82.0
4,105,HR,NaN,87.0


In [7]:
assert_frame_equal(pandas_selected, tidypy_selected)
print("Column selection matches.")


Column selection matches.


## 4. Compare batch mutation and bulk renaming

This is where `mutate_across()` and `rename_with()` are easiest to appreciate.


In [8]:
pandas_cleaned = (
    df.loc[:, ["employee_id", "dept", "score_math", "score_eng", "name"]]
      .assign(
          score_math=lambda x: x["score_math"].fillna(x["score_math"].median()),
          score_eng=lambda x: x["score_eng"].fillna(x["score_eng"].median()),
      )
      .rename(columns={"score_math": "math", "score_eng": "eng"})
      .assign(
          name=lambda x: x["name"].str.strip(),
          level=lambda x: pd.Series(
              ["A" if v >= 90 else "B" if v >= 80 else "C" for v in x["math"]],
              index=x.index,
          ),
      )
)

pandas_cleaned


,employee_id,dept,math,eng,name,level
0,101,Sales,92.0,85.0,Alice,A
1,102,Sales,88.0,90.0,Bob,B
2,103,Tech,88.0,86.0,Carol,B
3,104,Tech,79.0,82.0,David,C
4,105,HR,88.0,87.0,Frank,B


In [9]:
tidypy_cleaned = (
    df
    .pipe(
        select,
        "employee_id",
        "dept",
        starts_with("score_"),
        "name",
    )
    .pipe(
        mutate_across,
        starts_with("score_"),
        lambda s: s.fillna(s.median()),
    )
    .pipe(
        rename_with,
        lambda c: c.replace("score_", ""),
        starts_with("score_"),
    )
    .assign(
        name=lambda x: x["name"].str.strip(),
        level=lambda x: case_when(
            (x["math"] >= 90, "A"),
            (x["math"] >= 80, "B"),
            default="C",
        ),
    )
)

tidypy_cleaned


,employee_id,dept,math,eng,name,level
0,101,Sales,92.0,85.0,Alice,A
1,102,Sales,88.0,90.0,Bob,B
2,103,Tech,88.0,86.0,Carol,B
3,104,Tech,79.0,82.0,David,C
4,105,HR,88.0,87.0,Frank,B


In [10]:
assert_frame_equal(pandas_cleaned, tidypy_cleaned)
print("Cleaning pipeline matches.")


Cleaning pipeline matches.


## 5. Compare grouped summaries


In [11]:
pandas_summary = (
    tidypy_cleaned.groupby("dept", sort=False)
    .agg(
        avg_math=("math", "mean"),
        avg_eng=("eng", "mean"),
        n=("employee_id", "count"),
    )
    .reset_index()
)

pandas_summary


,dept,avg_math,avg_eng,n
0,Sales,90.0,87.5,2
1,Tech,83.5,84.0,2
2,HR,88.0,87.0,1


In [12]:
tidypy_summary = summarize(
    tidypy_cleaned,
    by="dept",
    avg_math=("math", "mean"),
    avg_eng=("eng", "mean"),
    n=("employee_id", "count"),
)

tidypy_summary


,dept,avg_math,avg_eng,n
0,Sales,90.0,87.5,2
1,Tech,83.5,84.0,2
2,HR,88.0,87.0,1


In [13]:
assert_frame_equal(pandas_summary, tidypy_summary)
print("Summary matches.")


Summary matches.


## 6. What this API group is really solving

These APIs are not trying to replace pandas. They are trying to give a more stable entry point for the kinds of cleaning steps that are easiest to repeat and scatter.


## Exercise

Add one more score column that starts with `score_`, then rerun both versions. Check whether the selector-based version needs almost no changes.
